In [1]:
import torch
import torch.nn as nn

In [2]:
# 生成一个4行五列的张量
x = torch.randn(4, 5)
print(x)

tensor([[ 0.0441,  0.1094,  1.4352,  2.1974,  1.7414],
        [-0.3264, -1.2155,  1.4996, -0.0337, -1.4412],
        [ 2.9182, -0.4919, -0.8950,  0.7946,  0.1296],
        [ 0.4743,  0.5962, -2.0117, -1.0121, -0.9844]])


In [3]:
# 打印后两列
print(x[:, -2:])

tensor([[ 2.1974,  1.7414],
        [-0.0337, -1.4412],
        [ 0.7946,  0.1296],
        [-1.0121, -0.9844]])


In [ ]:
# 交换两个维度
y = x.transpose(0, 1)
print(y)

tensor([[ 0.0441, -0.3264,  2.9182,  0.4743],
        [ 0.1094, -1.2155, -0.4919,  0.5962],
        [ 1.4352,  1.4996, -0.8950, -2.0117],
        [ 2.1974, -0.0337,  0.7946, -1.0121],
        [ 1.7414, -1.4412,  0.1296, -0.9844]])


In [6]:
x = torch.randn(3, 4, 4)
y = torch.randn(3, 4, 5)
# 求x y矩阵乘法
z = torch.matmul(x, y)
print(z.shape)

torch.Size([3, 4, 5])


In [7]:
# 两道问答题
# 1. 处理matmul, 还有哪些矩阵乘法操作
# - @, *

# 2. transpose, view, reshape, flatten各有什么用

# 3. view和reshape区别是什么，view什么时候会遇到非连续张量


In [13]:
# 写一个MLP拟合x和y
device = torch.device('cuda')
N, D_in, H, D_out = 64, 1000, 100, 10
x = torch.randn(N, D_in, device = device) # 输入
y = torch.randn(N, D_out, device = device) # label

class SimpleMlp(nn.Module):
    def __init__(self, hidden_dim, num_class=10, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.model = nn.Sequential( # [N, D_in]
            nn.Linear(1000, hidden_dim), # [N, hidden_dim]
            nn.ReLU(),
            nn.Linear(hidden_dim, num_class) # [N, D_out]
        )
        
    
    def forward(self, x:torch.tensor):
        return self.model(x)

In [14]:
def train(model, optim, x, label, epochs=50):
    # criteria = nn.CrossEntropyLoss() # 这里用错损失函数了，y不是分类任务
    criteria = nn.MSELoss()
    for epoch in range(epochs):

        output = model(x) # [N, D_out]
        loss =  criteria(output, label)

        if epoch % 10 == 0:
            print(f"epoch: {epoch}, loss: {loss}")

        loss.backward()
        with torch.no_grad():
            optim.step()
            optim.zero_grad()

In [ ]:
model = SimpleMlp(hidden_dim=H, num_class=D_out).to(device)
optim = torch.optim.Adam(model.parameters(), lr=0.001) # optim不需要to(device)
x = x.to(device)
y = y.to(device)

# 面试的时候用错损失函数了，loss没有收敛
train(model, optim, x, y)

epoch: 0, loss: 1.238726019859314
epoch: 10, loss: 0.11535065621137619
epoch: 20, loss: 0.03502105548977852
epoch: 30, loss: 0.011400274932384491
epoch: 40, loss: 0.004351056646555662


In [16]:
# 实现自注意力层
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = self.d_model // self.num_heads

        self.qkv_proj =  nn.Linear(d_model, 3* self.hidden_dim * num_heads) 
        self.final_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        # input: [Batch, seq_len, d_model]
        # 计算QKV [Batch, seq_len, hidden_dim] * 3 * num_heads
        batch, seq_len, d_model = x.shape

        qkv = self.qkv_proj(x) # [B, seq_len, hidden_dim* 3 * num_heads]
        qkv = qkv.reshape(batch, seq_len, 3, self.num_heads, -1)

        # 分离qkv
        qkv = qkv.permute(2, 0, 3, 1, 4) 
        q, k, v = qkv[0], qkv[1], qkv[2] # [batch, num_heads, seq_len, hidden_dim]

        # 计算注意力分数
        score = torch.matmul(q, k.transpose(-2,-1)) / self.hidden_dim # [batch, num_heads ,seq_len, seq_len ]

        # 计算注意力值
        attention_value = torch.softmax(score, dim=-1) @ v # [batch, num_heads, seq_len, hidden_dim]

        # 堆叠头
        attention_value = attention_value.transpose(1, 2) # [batch, seq_len, num_heads, hidden_dim]
        attention_value = attention_value.reshape(batch, seq_len, d_model)

        # 投影层
        return self.final_proj(attention_value)

In [17]:
model = MultiHeadAttention(d_model=512, num_heads=4)
x = torch.randn(3, 5, 512) # [batch, seq_len , d_model]
y = model(x)
print(y.shape == x.shape)

True
